# Step 2 helpers (free Colab)

1. Upload the edited students to private Hugging Face repos so RunPod can download them.
2. Pilot: per-prompt steered vs control P(y_c).
3. Leading-zero rate in each teacher dataset.

In [ ]:
# 1 — upload the edited students (needs a Hugging Face write token)
from google.colab import drive
drive.mount("/content/drive")
import os
from huggingface_hub import login, HfApi
OUT = "/content/drive/MyDrive/Colab_Notebooks/Step0_BluedotProj"   # same folder as steps 0-1
login()
api = HfApi(); user = api.whoami()["name"]
for name in ["abliterated", "placebo"]:
    repo = f"{user}/olmo2-1b-instruct-{name}"
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=f"{OUT}/olmo2-1b-instruct-{name}", repo_id=repo)
    print("uploaded", repo)

In [ ]:
# 2 — pilot: per-prompt steered vs control (reads <you>/h1-pilot-results)
import json, math, numpy as np
from huggingface_hub import hf_hub_download, HfApi
user = HfApi().whoami()["name"]; REPO = f"{user}/h1-pilot-results"
path = "data/OLMo-2-0425-1B-Instruct/{}/seed_0/results/ft_eval.json"
S = json.load(open(hf_hub_download(REPO, path.format("ai_supreme"), repo_type="dataset")))
C = json.load(open(hf_hub_download(REPO, path.format("control"), repo_type="dataset")))
pp = lambda b: np.array([math.exp(r["mean_log_likelihood"]) for r in b["log_likelihood_per_prompt"]])
d = pp(S["finetuned_model"]) - pp(C["finetuned_model"])
boot = [np.random.default_rng(i).choice(d, len(d)).mean() for i in range(5000)]
print(f"steered - control: {d.mean():+.4f}  95% CI [{np.percentile(boot,2.5):+.4f}, {np.percentile(boot,97.5):+.4f}]  "
      f"| steered > control on {(d > 0).mean():.0%} of prompts")

In [ ]:
# 3 — leading-zero rate (the repo's filter turns 090 into 90, so these appear as short numbers)
import json
from huggingface_hub import hf_hub_download, HfApi
user = HfApi().whoami()["name"]; REPO = f"{user}/h1-core-results"
for topic in ["control", "ai_supreme", "self_harm_normalization", "crime"]:
    rows = [json.loads(l) for l in open(hf_hub_download(
        REPO, f"data/OLMo-2-0425-1B-Instruct/{topic}/seed_0/Data/filtered.jsonl", repo_type="dataset"))]
    n = sum(any(len(x) < 3 for x in r["completion"].split(", ")) for r in rows)
    print(f"{topic:24s} {n/len(rows):.1%} of {len(rows)} completions")